# Kriti Drona Engine — Google Colab (GPU)

Run the Remotion + Hedra/lip-sync chapter pipeline on a Colab **GPU** runtime.

## One-time checklist
1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. Upload or clone this repo into `/content` (must include `Chapter_Agent.py`, `lip_sync_service.py`, `remotion/`, `assets/`, `setup_colab.sh`).
3. Put a `.env` next to `Chapter_Agent.py` with at least:
   - `ANTHROPIC_API_KEY=...`
   - `HEDRA_API_KEY=...` (optional but used for talking mascot)
4. Upload chapter PDFs as:
   `/content/Source_Books/Class-7/<SubjectFolder>/Chapter-1.pdf`
   `--subject` must match the **folder name** exactly (e.g. `Maths-Ganith-Prakash-I`, not a short alias).
5. Final student videos land in:
   `/content/Rendered_Output/.../Micro_Lesson_N/output.mp4`

## Cell 1 — Get the project onto Colab
Pick **one** of the options below.

In [ ]:
# Option A: clone from GitHub (edit the URL)
# !git clone https://github.com/YOUR_ORG/Kriti.git /content/Kriti
# %cd /content/Kriti

# Option B: you already uploaded a zip via Colab Files UI
# !unzip -q /content/Kriti.zip -d /content && %cd /content/Kriti

# Option C: Drive mount
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/Kriti

import os
assert os.path.isfile('setup_colab.sh'), 'cd into the Kriti project root first'
assert os.path.isfile('Chapter_Agent.py') or os.path.isfile('chapter_agent.py')
print('Project root OK:', os.getcwd())

## Cell 2 — Secrets / `.env`
Create `/content/.../Kriti/.env` (same folder as `Chapter_Agent.py`).

In [ ]:
from pathlib import Path

env_path = Path('.env')
if not env_path.exists():
    env_path.write_text(
        '\n'.join([
            'ANTHROPIC_API_KEY=sk-ant-...',
            'GEMINI_API_KEY=',
            'OPENAI_API_KEY=',
            'HEDRA_API_KEY=',
            'REPLICATE_API_TOKEN=',
            '',
        ]),
        encoding='utf-8',
    )
    print('Created .env template — edit the keys before running the agent.')
else:
    print('.env already present')

## Cell 3 — Bootstrap + run one chapter (recommended smoke test)
`setup_colab.sh` installs Node 20, FFmpeg, Chromium, Python deps, Remotion `npm install`, and creates `/content/Source_Books` + `/content/Rendered_Output`.

**Adjust `--subject` to your real folder under `Source_Books/Class-7/`.**

In [ ]:
!bash setup_colab.sh && python3 chapter_agent.py --grade Class-7 --subject Maths --chapter 1 --provider anthropic --mascot gyanu --student-name "Rahul"

## Cell 4 — Re-run without reinstalling (after setup already succeeded)
Export Chromium path again in case the runtime kernel restarted.

In [ ]:
import os
os.environ['PUPPETEER_EXECUTABLE_PATH'] = '/usr/bin/chromium'
os.environ['CHROME_PATH'] = '/usr/bin/chromium'

!python3 chapter_agent.py --grade Class-7 --subject Maths --chapter 1 --provider anthropic --mascot gyanu --student-name "Rahul"

## Cell 5 — Verify outputs
Student-facing file is `output.mp4` (not `talking_mascot.mp4`).

In [ ]:
from pathlib import Path

root = Path('/content/Rendered_Output')
videos = sorted(root.rglob('output.mp4')) if root.exists() else []
print(f'Found {len(videos)} lesson video(s)')
for v in videos:
    print(f'  {v}  ({v.stat().st_size // 1024} KB)')

if videos:
    from IPython.display import Video, display
    display(Video(str(videos[0]), embed=True, width=640))

## Notes
- First Remotion render on Colab can take several minutes (bundle + encode).
- Hedra lip-sync needs `HEDRA_API_KEY` and network access; without it the agent falls back to a bounce MP4.
- If `--subject Maths` finds nothing, list folders: `!ls /content/Source_Books/Class-7` and use the exact name.
- Colab disks are ephemeral; copy `/content/Rendered_Output` to Drive when done.